# 03 · Results for the paper

- **A.** SmolLM2 baseline (CausalCoT run on the easy file), rescored with the tolerant yes/no parser.
- **B.** Commonsense-alignment results from notebook 02: accuracy by variant (H1), P(Yes) shift on matched pairs (H1b), accuracy by rung (H2), answer bias, topology, failure graphs.
- **C.** Export: LaTeX table rows and PDF figures into the paper folder.
- **D.** CausalCoT vs direct on the same items: hypotheses, truncated reasoning, trace reader, export.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from causal_evaluation.evaluation.failure_graph import draw_failure_graph, failure_graph
from causal_evaluation.evaluation.stats import extract_yes_no
from causal_evaluation.evaluation.variant_analysis import (
    load_scored,
    nonsense_gap,
    p_yes_shift,
    paired_accuracy_gap,
    rate_table,
)

INTERIM = Path(os.environ.get("CE_INTERIM", ROOT / "data" / "interim"))
PAPER_DIR = Path(os.environ.get("CE_PAPER_DIR", ROOT.parent / "tesispp" / "cadis-extended-abstract"))
FIG_DIR, TAB_DIR = PAPER_DIR / "figures", PAPER_DIR / "tables"
for folder in (FIG_DIR, TAB_DIR):
    folder.mkdir(parents=True, exist_ok=True)
pd.set_option("display.precision", 3)


def short(model):
    return model.split("/")[-1].split(":")[0]

## A · SmolLM2 baseline, rescored

In [ ]:
sample_full = pd.read_json(INTERIM / "sample-full.jsonl", lines=True, dtype={"question_id": str})
responses = pd.read_json(INTERIM / "responses-smollm2-full.jsonl", lines=True, dtype={"question_id": str})
responses["text"] = responses["response"].map(lambda r: r.get("text") or "")
responses["error"] = responses["response"].map(lambda r: r.get("error"))

base = responses.merge(
    sample_full[["question_id", "answer", "causal_level", "meta"]], on="question_id", validate="one_to_one"
)
base["answer"] = base["answer"].str.strip().str.lower()
base["story_kind"] = base["meta"].map(lambda m: "nonsense" if str(m["story_id"]).startswith("nonsense") else "named")
base["parsed"] = base["text"].map(extract_yes_no)
base["status"] = np.select([base["error"].notna(), base["parsed"].isna()], ["error", "unparsed"], "parsed")
base["correct_exact"] = base["text"].str.strip().str.casefold() == base["answer"]
base["correct_parsed"] = base["parsed"] == base["answer"]
base["is_yes"] = base["parsed"] == "yes"

base["status"].value_counts()

In [ ]:
baseline = pd.DataFrame(
    {
        "exact_match": base.groupby("causal_level")["correct_exact"].mean(),
        "tolerant_parser": base.groupby("causal_level")["correct_parsed"].mean(),
        "n": base.groupby("causal_level").size(),
    }
)
baseline.loc["overall"] = [base["correct_exact"].mean(), base["correct_parsed"].mean(), len(base)]
baseline

In [ ]:
parsed = base[base["status"] == "parsed"]
display(rate_table(parsed, ["causal_level"], flag="correct_parsed"))
display(rate_table(parsed, ["story_kind"], flag="correct_parsed"))
print(f"Yes rate among parsed answers: {parsed['is_yes'].mean():.3f} (answers are 50/50)")

## B · Commonsense alignment

In [ ]:
USE_SMOKE = False  # True only to test this notebook with the -smoke files from notebook 02
result_paths = [
    p
    for p in sorted(INTERIM.glob("logprobs-*.jsonl"))
    if p.name.endswith("-smoke.jsonl") == USE_SMOKE and "causal_cot" not in p.name
]
print("\n".join(p.name for p in result_paths))
df = load_scored(INTERIM / "variants-sample.jsonl", result_paths)
df["is_yes"] = df["pred"] == "yes"
df.groupby("model").agg(
    items=("item_id", "size"),
    median_answer_mass=("answer_mass", "median"),
    yes_rate=("is_yes", "mean"),
)

### Accuracy by variant

In [ ]:
acc = rate_table(df, ["model", "variant"])
acc.pivot(index="model", columns="variant", values="rate")

### H1 · Anticommonsense gap on matched pairs, and nonsense gap

In [ ]:
h1_paired = paired_accuracy_gap(df)
h1_paired

In [ ]:
h1_nonsense = nonsense_gap(df)
h1_nonsense

### H1b · P(Yes) shift on matched relational items (commonsense minus anticommonsense)

In [ ]:
h1b = p_yes_shift(df)
h1b

### H2 · Accuracy by rung

In [ ]:
h2 = rate_table(df, ["model", "rung"])
h2.pivot(index="model", columns="rung", values="rate")

### Answer bias and topology

In [ ]:
yes_rates = rate_table(df, ["model", "variant"], flag="is_yes")
display(yes_rates.pivot(index="model", columns="variant", values="rate"))
rate_table(df, ["model", "graph_id"]).pivot(index="graph_id", columns="model", values="rate")

## C · Figures and table for the paper

In [ ]:
order = ["commonsense", "noncommonsense", "anticommonsense"]
models = sorted(df["model"].unique())
fig, ax = plt.subplots(figsize=(6.5, 3))
width = 0.8 / len(order)
for i, variant in enumerate(order):
    rows = acc[acc["variant"] == variant].set_index("model").reindex(models)
    x = np.arange(len(models)) + (i - 1) * width
    ax.bar(x, rows["rate"], width, label=variant,
           yerr=[rows["rate"] - rows["lo"], rows["hi"] - rows["rate"]], capsize=2)
ax.axhline(0.5, color="gray", lw=0.8, ls="--")
ax.set_xticks(np.arange(len(models)), [short(m) for m in models], rotation=15, fontsize=7)
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
ax.legend(fontsize=7, frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "accuracy_by_variant.pdf")

In [ ]:
for model in models:
    nodes, edges = failure_graph(df[df["model"] == model])
    fig, ax = plt.subplots(figsize=(7, 4))
    draw_failure_graph(nodes, edges, ax=ax, title=short(model))
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"failure_graph_{short(model)}.pdf")
    display(edges.sort_values("lift", ascending=False).head(8))

In [ ]:
def cell(row):
    return f"{row.rate:.2f} [{row.lo:.2f}, {row.hi:.2f}]"


shift_yes = h1b[h1b["answer"] == "yes"].set_index("model")
lines = []
for model in models:
    cells = {r.variant: cell(r) for r in acc[acc["model"] == model].itertuples()}
    p_yes = (
        f"{shift_yes.loc[model, 'p_yes_common']:.2f} / {shift_yes.loc[model, 'p_yes_anti']:.2f}"
        if model in shift_yes.index
        else "--"
    )
    name = short(model).replace("_", r"\_")
    lines.append(
        f"{name} & {cells.get('commonsense', '--')} & {cells.get('noncommonsense', '--')} & "
        f"{cells.get('anticommonsense', '--')} & {p_yes} \\\\"
    )
(TAB_DIR / "variants_rows.tex").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

## D · CausalCoT vs direct on the same items

The CausalCoT run covers a balanced subset (150 matched pairs + 150 nonsense items per model). Every comparison here restricts the direct results to those same items. Model labels read `<model> | direct` and `<model> | causal_cot`.

Result files used:

In [ ]:
SAMPLE_PATH = INTERIM / "variants-sample.jsonl"
DIRECT_PATHS = result_paths
COT_PATHS = [p for p in sorted(INTERIM.glob("logprobs-*-causal_cot*.jsonl")) if p.name.endswith("-smoke.jsonl") == USE_SMOKE]
for path in [SAMPLE_PATH, *DIRECT_PATHS, *COT_PATHS]:
    print(path)

both = load_scored(SAMPLE_PATH, DIRECT_PATHS + COT_PATHS)
both["base"] = both["model"].str.replace("-causal_cot", "", regex=False).map(short)
both["strategy"] = np.where(both["model"].str.endswith("-causal_cot"), "causal_cot", "direct")
cot_ids = set(both.loc[both["strategy"] == "causal_cot", "item_id"])
same = both[both["item_id"].isin(cot_ids)].copy()
same["model"] = same["base"] + " | " + same["strategy"]
same["is_yes"] = same["pred"] == "yes"
cot = same[same["strategy"] == "causal_cot"].copy()
same.groupby("model").size()

### Accuracy by variant

In [ ]:
acc_same = rate_table(same, ["model", "variant"])
acc_same.pivot(index="model", columns="variant", values="rate")

### H1 · anticommonsense gap on matched pairs, and nonsense gap

In [ ]:
h1_same = paired_accuracy_gap(same)
h1_same

In [ ]:
nonsense_gap(same)

### H1b · P(Yes) shift on matched relational items (commonsense minus anticommonsense)

In [ ]:
p_yes_shift(same)

### H2 (by rung) and answer bias

In [ ]:
display(rate_table(same, ["model", "rung"]).pivot(index="model", columns="rung", values="rate"))
rate_table(same, ["model", "variant"], flag="is_yes").pivot(index="model", columns="variant", values="rate")

### Truncated reasoning

`reasoning_truncated` means the trace hit `max_new_tokens` before the model finished, so the final Yes/No was given on incomplete reasoning. If truncation is similar across variants it does not bias H1, but it lowers accuracy.

In [ ]:
display(cot.pivot_table(index="base", columns="variant", values="reasoning_truncated", aggfunc="mean"))
display(cot.pivot_table(index="query_type", columns="base", values="reasoning_truncated", aggfunc="mean"))
cot.pivot_table(index="base", columns="reasoning_truncated", values="correct", aggfunc=["mean", "size"])

### Finished traces only (secondary check)

Keeps pairs where **both** members finished their reasoning. This is not a random subset (harder items truncate more often), so treat it as a robustness check, not the main result.

In [ ]:
finished = cot[~cot["reasoning_truncated"].astype(bool)]
pair_size = finished.groupby(["model", "pair_id"])["item_id"].transform("size")
display(paired_accuracy_gap(finished[pair_size == 2]))
rate_table(finished, ["model", "variant"]).pivot(index="model", columns="variant", values="rate")

### Reading reasoning traces

`show_trace` prints the prompt the model saw, its reasoning, P(Yes) and the correct answer. Filter by model (substring), variant, correctness or truncation. `show_pair` prints both members of a matched pair for one model, to compare how the renamed variable changed the reasoning.

In [ ]:
from causal_evaluation.models.logprob import item_text


def _print_row(row, max_chars):
    print(f"=== {row['model']} | {row['item_id']} | {row['query_type']} (rung {row['rung']}) | "
          f"correct answer: {row['answer']} | P(Yes) = {row['p_yes']:.2f} | "
          f"{'CORRECT' if row['correct'] else 'WRONG'}{' | TRUNCATED' if row['reasoning_truncated'] else ''}")
    print("--- prompt\n" + item_text(row))
    print("--- reasoning\n" + str(row["reasoning"])[:max_chars] + "\n")


def show_trace(model="", variant=None, correct=None, truncated=None, n=1, seed=0, max_chars=3000):
    rows = cot[cot["model"].str.contains(model, regex=False)]
    if variant is not None:
        rows = rows[rows["variant"] == variant]
    if correct is not None:
        rows = rows[rows["correct"] == correct]
    if truncated is not None:
        rows = rows[rows["reasoning_truncated"].astype(bool) == truncated]
    print(f"{len(rows)} matching traces")
    for _, row in rows.sample(min(n, len(rows)), random_state=seed).iterrows():
        _print_row(row, max_chars)


def show_pair(model, pair_id=None, seed=0, max_chars=2000):
    rows = cot[cot["model"].str.contains(model, regex=False) & cot["pair_id"].notna()]
    if pair_id is None:
        pair_id = rows["pair_id"].sample(1, random_state=seed).iloc[0]
    for _, row in rows[rows["pair_id"] == pair_id].sort_values("variant", ascending=False).iterrows():
        _print_row(row, max_chars)


show_trace("Qwen2.5-7B", variant="anticommonsense", correct=False)

In [ ]:
show_pair("Phi-3.5")

### Made-up data (heuristic)

Deterministic counterfactuals give no probabilities in the prompt. A trace that nonetheless writes probability values such as `P(...) = 0.3` has invented data. This is a rough text filter, so read the flagged traces before quoting a rate.

In [ ]:
cot["prompt_has_numbers"] = cot["given_info"].str.contains(r"\d")
cot["trace_states_probabilities"] = cot["reasoning"].fillna("").str.contains(r"P\([^)]*\)\s*=\s*0?\.\d")
cot["invented_numbers"] = ~cot["prompt_has_numbers"] & cot["trace_states_probabilities"]
no_numbers = cot[~cot["prompt_has_numbers"]]
display(no_numbers.groupby(["base", "query_type"])["invented_numbers"].agg(["mean", "size"]))
flagged = cot[cot["invented_numbers"]]
if len(flagged):
    _print_row(flagged.iloc[0], 2500)

### Export: direct vs CausalCoT figure and table rows

In [ ]:
bases = sorted(same["base"].unique())
acc_overall = rate_table(same, ["base", "strategy"])
fig, ax = plt.subplots(figsize=(6.5, 3))
for i, strategy in enumerate(["direct", "causal_cot"]):
    rows = acc_overall[acc_overall["strategy"] == strategy].set_index("base").reindex(bases)
    x = np.arange(len(bases)) + (i - 0.5) * 0.38
    ax.bar(x, rows["rate"], 0.38, label="CausalCoT" if strategy == "causal_cot" else "direct",
           yerr=[rows["rate"] - rows["lo"], rows["hi"] - rows["rate"]], capsize=2)
ax.axhline(0.5, color="gray", lw=0.8, ls="--")
ax.set_xticks(np.arange(len(bases)), bases, rotation=15, fontsize=7)
ax.set_ylabel("Accuracy (same 450 items)")
ax.set_ylim(0, 1)
ax.legend(fontsize=7, frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "accuracy_direct_vs_cot.pdf")

In [ ]:
gaps = h1_same.set_index("model")
trunc = cot.groupby("base")["reasoning_truncated"].mean()
rows_tex = []
for base in bases:
    d, c = gaps.loc[f"{base} | direct"], gaps.loc[f"{base} | causal_cot"]
    acc_d = acc_overall.set_index(["base", "strategy"]).loc[(base, "direct"), "rate"]
    acc_c = acc_overall.set_index(["base", "strategy"]).loc[(base, "causal_cot"), "rate"]
    name = base.replace("_", r"\_")
    rows_tex.append(
        f"{name} & {acc_d:.2f} & {acc_c:.2f} & "
        f"{d.gap:+.3f} [{d.gap_lo:+.2f}, {d.gap_hi:+.2f}] & "
        f"{c.gap:+.3f} [{c.gap_lo:+.2f}, {c.gap_hi:+.2f}] & {trunc[base]:.2f} \\\\"
    )
(TAB_DIR / "cot_rows.tex").write_text("\n".join(rows_tex) + "\n")
print("columns: model & acc direct & acc CausalCoT & gap direct [CI] & gap CausalCoT [CI] & truncated")
print("\n".join(rows_tex))